# أرشيف منهجية التدريب والاستعادة السابقة

هذا دفتر تاريخي محفوظ لصون قابلية تتبع الشيفرة واختباراتها، وليس مسار التدريب النهائي أو التشغيل الموصى به. التجربة النهائية المستقلة موثقة في [دفتر 1200 خطوة](Arabic_News_Training_1200.ipynb)، والتشغيل في [الدفتر النهائي](Khulasa_Final_Demo.ipynb).

تستخدم الخطة التاريخية AraT5v2 وArabic XL-Sum وتنفذ اختبارًا أوليًا وخط أساس وتجربتي ضبط مع حفظ حالة التدريب. تختلف عن التجربة النهائية المستقلة في التجميع والعينات والاستعادة؛ لا يجوز إسناد نتائجها إلى النموذج النهائي دون تطابق البروتوكول.


In [ ]:
import os
import platform
import subprocess
from pathlib import Path

import torch

print("Python:", platform.python_version(), "PyTorch:", torch.__version__)
if not torch.cuda.is_available():
    raise RuntimeError("Enable a T4 GPU using Runtime > Change runtime type, then run all again.")
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM GiB:", round(torch.cuda.get_device_properties(0).total_memory / 2**30, 2))
subprocess.run(["nvidia-smi"], check=True)


## التخزين الدائم وأنماط التنفيذ

السقف التاريخي 40 GB مع احتياط checkpoints قدره 7 GB، ولا يمثل مساحة ممنوحة من Drive. يقتضي الاستئناف ثبات النمط ومسار التشغيل وإعدادات التجربة. تغيير المعلمات يتطلب تجربة جديدة، ولا يجوز تشغيل جلستين على مجلد واحد. تُضبط `MODE` و`CODE_REF` و`SMOKE_ONLY` و`RUN_NAME` في خلية الإعداد الوحيدة.


In [ ]:
from google.colab import drive

drive.mount("/content/drive")

# EDIT RETRY SETTINGS HERE ONLY. Later cells do not overwrite these variables.
MODE = os.environ.get("SUMMARY_MODE", "full")  # use "dev" for the first corrected smoke
CODE_REF = os.environ.get("SUMMARY_CODE_REF", "main")  # fixed SHA or PR branch before merge
SMOKE_ONLY = os.environ.get("SUMMARY_SMOKE_ONLY", "0") == "1"  # True for first retry
RUN_NAME = os.environ.get("SUMMARY_RUN_NAME", MODE + "-v2-numerics")
if MODE not in {"dev", "full"}:
    raise ValueError("SUMMARY_MODE must be dev or full")
if not RUN_NAME or Path(RUN_NAME).name != RUN_NAME or RUN_NAME in {".", ".."}:
    raise ValueError("RUN_NAME must be a single safe directory name")
RUN_DIR = Path("/content/drive/MyDrive/AUTHORIZED_PROJECT") / RUN_NAME
RUN_DIR.mkdir(parents=True, exist_ok=True)
print({"mode": MODE, "requested_source": CODE_REF, "smoke_only": SMOKE_ONLY,
       "persistent_run_directory": str(RUN_DIR)})
print("Preserve old experiments and source_commit.json files. Use a fresh directory for this correction.")


## تثبيت مصدر الخطة التاريخية

تُحفظ بصمة المصدر عند بدء التشغيل، ثم تُستعاد عند الاستئناف. أي مرجع قديم في الخلايا جزء من السجل التاريخي؛ لا يُستخدم لتشغيل التطبيق النهائي. تُثبت المكتبات الحساسة للواجهات، ولا تُنفذ شيفرة تحميل بيانات بعيدة.


In [ ]:
import ast
import json
import re
import sys


def checkout_source(repository, ref, repo_dir, run_dir, expected_schema=2):
    """Freeze an explicitly fetched ref; never reuse stale HEAD for a new run."""
    repo_dir, run_dir = Path(repo_dir), Path(run_dir)
    run_dir.mkdir(parents=True, exist_ok=True)
    new_checkout = not (repo_dir / ".git").exists()
    if new_checkout:
        subprocess.run(["git", "clone", "--no-checkout", repository, str(repo_dir)], check=True)
    origin = subprocess.check_output(["git", "-C", str(repo_dir), "remote", "get-url", "origin"], text=True).strip()
    if origin != repository:
        raise RuntimeError("Existing code checkout belongs to another repository; use a fresh runtime.")
    dirty = subprocess.check_output(["git", "-C", str(repo_dir), "status", "--porcelain"], text=True).strip()
    if dirty and not new_checkout:
        raise RuntimeError("Code checkout has local edits. Preserve them and use a fresh runtime.")
    source_lock = run_dir / "source_commit.json"
    locked = json.loads(source_lock.read_text()) if source_lock.exists() else None
    target = locked["commit"] if locked else ref
    if locked and (locked.get("repository") != repository or not re.fullmatch(r"[0-9a-f]{40}", target)):
        raise RuntimeError("Invalid source lock; preserve it and choose a new run directory.")
    subprocess.run(["git", "-C", str(repo_dir), "fetch", "origin", target], check=True)
    commit = subprocess.check_output(["git", "-C", str(repo_dir), "rev-parse", "FETCH_HEAD"], text=True).strip()
    if locked and commit != target:
        raise RuntimeError("Fetched commit does not match the persistent source lock.")
    subprocess.run(["git", "-C", str(repo_dir), "checkout", "--detach", commit], check=True)
    # Validate the checked-out source without importing it or pinning an old main.
    config_path = repo_dir / "src/arabic_summarizer/config.py"
    schema = None
    if config_path.is_file():
        tree = ast.parse(config_path.read_text(encoding="utf-8"))
        for node in tree.body:
            if isinstance(node, ast.ClassDef) and node.name == "Config":
                for field in node.body:
                    if (isinstance(field, ast.AnnAssign) and isinstance(field.target, ast.Name)
                            and field.target.id == "training_schema_version"):
                        schema = ast.literal_eval(field.value)
    if schema != expected_schema:
        pin_status = "Existing source lock was preserved." if locked else "No new source lock was written."
        raise RuntimeError(
            f"Incompatible source: requested_ref={ref}, fetched_commit={commit}, "
            f"training_schema={schema}, expected={expected_schema}, config_file={config_path}. "
            f"{pin_status} PR #2 may not be merged into main. In the SETUP cell select the fixed "
            "commit 563378c795da2ac483b9725c1c1d5486ef880622 and a NEW empty run directory. "
            "Do not delete or rewrite old source_commit.json files."
        )
    if not locked:
        temporary = source_lock.with_suffix(".tmp")
        temporary.write_text(json.dumps({"commit": commit, "repository": repository}))
        temporary.replace(source_lock)
    return commit


REPOSITORY = "https://github.com/nada-ali-13/Arabic-news-summarizer.git"
REPO_DIR = Path("/content/Arabic-news-summarizer")
if any(name == "arabic_summarizer" or name.startswith("arabic_summarizer.") for name in sys.modules):
    raise RuntimeError("Project modules are already loaded. Restart the Colab runtime to load the selected commit safely.")
commit = checkout_source(REPOSITORY, CODE_REF, REPO_DIR, RUN_DIR)
print("Source schema verified: 2; source commit:", commit)
print("Requested ref:", CODE_REF, "(an existing source_commit.json takes precedence)")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO_DIR / "requirements.txt")], check=True)
sys.path.insert(0, str(REPO_DIR))
sys.path.insert(0, str(REPO_DIR / "src"))
os.chdir(REPO_DIR)
freeze = subprocess.check_output([sys.executable, "-m", "pip", "freeze"], text=True)
(RUN_DIR / "installed_packages.txt").write_text(freeze)


## تنفيذ الخطة التاريخية أو استعادتها

يشمل `dev` خطوتي smoke وعشر خطوات تطوير. يشمل `full` اختبارًا أوليًا وتجربتين مستقلتين حتى 1200 خطوة بمعدلي تعلم 5e-5 و1e-4، وbatch size يساوي 1 وتجميع 16. الحد 1024 token للمصدر و128 للهدف، والإيقاف المبكر على validation ROUGE-Lsum بصبر 3. تختلف هذه الإعدادات عن التجربة النهائية ذات التجميع 8.

تُثبت عينات validation للمقارنة؛ يُقيّم خط الأساس قبل الضبط، ويُحفظ test إلى انتهاء الخطة. ميزانية الجلسة 3 ساعات والتراكمية 7 ساعات، مع احتمال تجاوز حدود الفحص بسبب التقييم وحفظ الملفات.


In [ ]:
from arabic_summarizer.config import load_config
from arabic_summarizer.io import atomic_json
from arabic_summarizer.training import run

config = load_config(REPO_DIR / "configs" / (MODE + ".json"))
# Optional budget change on restart, without changing experiment semantics:
# from dataclasses import replace
# config = replace(config, max_hours=10, session_hours=4)
assert getattr(config, "training_schema_version", None) == 2, "Old source pinned: use the fixed ref and a new run directory."
print(json.dumps(config.to_dict(), indent=2, ensure_ascii=False))
result = run(config, RUN_DIR, smoke_only=SMOKE_ONLY)
print(json.dumps(result, indent=2, ensure_ascii=False))
atomic_json(RUN_DIR / "notebook_last_result.json", result)


## فحص نتائج الخطة والعينات النوعية

لا تُملأ درجات قبل التنفيذ. يتوافر `final_report.json` بعد اكتمال الخطة والمقارنة. قد يشير `best.json` إلى تجربة مكتملة أثناء توقف تجربة أخرى. تُفحص العينات للحذف والتكرار والطلاقة واختلاق الوقائع.


In [ ]:
from IPython.display import display

for name in ["dataset_statistics.json", "experiments.json", "best.json", "final_report.json"]:
    path = RUN_DIR / name
    if path.exists():
        value = json.loads(path.read_text())
        print("\n", name)
        if name == "experiments.json":
            value = {k: {field: row.get(field) for field in ["status", "metrics", "best_checkpoint", "export"]}
                     for k, row in value.items()}
        display(value)
    else:
        print(name, "— pending")
examples_path = RUN_DIR / "best_test.json"
if examples_path.exists():
    examples = json.loads(examples_path.read_text())["examples"][:5]
    for example in examples:
        display({k: example[k] for k in ["id", "reference", "prediction"]})


## عرض تاريخي اختياري

تُحفظ التصديرات المستقلة في `RUN_DIR/exports/`، ويحدد `best.json` النموذج المختار ضمن هذه الخطة. لا تُستبدل به تلقائيًا أوزان النموذج النهائي الموثقة. المشاركة عبر Gradio عامة ومؤقتة، وعدد طلبات التوليد المتزامنة محدود.


In [ ]:
import gc

from app import build_app

best_path = RUN_DIR / "best.json"
if not best_path.exists():
    print("No completed fine-tuned export yet. A smoke-only run does not create one; no app is launched.")
else:
    os.environ["ARABIC_SUMMARIZER_MODEL"] = json.loads(best_path.read_text())["path"]
    gc.collect()
    torch.cuda.empty_cache()
    demo = build_app()
    demo.launch(share=True, debug=False)


## استئناف التدريب (resume) وحدوده التاريخية

تعيد الخطة استخدام النمط ومسار التشغيل وبصمات المصدر والبيانات والنموذج. تُتجاوز التجارب المكتملة، وتُستأنف أحدث حالة تدريب كاملة صالحة، مع الرجوع إلى حالة أقدم إذا فشل التحقق. إذا غابت حالة صالحة، يتوقف التنفيذ دون حذف الملفات.

لا يضمن Drive معاملات ذرية عبر الانقطاع. تكشف البصمات وعلامات الاكتمال الكتابات الجزئية، لكنها لا تستعيد ملفات محذوفة. لا يُعاد تفعيل هذه المنظومة ضمن دفتر العرض النهائي.
